# [Step 1 - Local-first chat models] Running Llama 3.2 on your own machine with Ollama

> **MLCourse - Agentic AI - Chat Models and Providers**

> Stage in the capstone: the generate stage - every answer your final RAG chatbot
> produces comes out of exactly the kind of chat model call you will run in this
> notebook, so mastering this one atom pays off in every later module.

### What you'll learn

- What a chat model call actually is: a list of typed messages in, exactly one assistant message out.
- How to run Meta's Llama 3.2 entirely on your laptop with LangChain's `ChatOllama` class - no key, no cost, no data leaving the machine.
- The three workhorse methods: `invoke()` for one answer, `stream()` for token-by-token output, and `batch()` for many prompts at once.
- What the `temperature` parameter really controls, verified with a hands-on experiment loop.
- A first taste of `init_chat_model()`, the provider-swapping factory used throughout the rest of the track.

> **Pro tip:** this is the only module where every demo runs 100 percent free and
> local. Break things on purpose here: kill the Ollama server mid-run, mistype model
> names, crank temperature to 2.0 - mistakes are the fastest teacher.

### Standard library imports


In [ ]:
import os                # Reads environment variables after load_dotenv() fills them.
import time              # Used to time responses so you can feel local latency yourself.
from pathlib import Path # Object-oriented paths; cleaner than string concatenation.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv  # Loads KEY=value lines from a .env file into os.environ.

# Walk up from wherever this notebook was launched until we reach the track root
# folder "03_agentic_ai". This makes the notebook runnable from any subfolder,
# because the .env file with provider keys lives at that root (it is gitignored).
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")     # No keys needed for Ollama, but later notebooks reuse this exact block.

# Jupyter plotting magic, wrapped so this file stays valid pure Python when it is
# executed as a plain script or checked by automated QA tooling outside IPython.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                        # Not running inside IPython - nothing to configure, move on.

print("Setup complete. Track root resolved to:", TRACK)


### 1. What exactly is a "chat model call"?

A chat model is a function with a tiny contract:

- **Input**: a LIST of messages. Each message carries a role plus text.
  Roles are `system` (persistent instructions from the developer), `human`
  (the end user's turn), and `ai` (the model's previous replies).
- **Output**: exactly ONE new `ai` message containing the reply text plus metadata.

The model never "remembers" anything between calls: if you want history in scope,
YOU put earlier messages back into the input list. That single fact explains why
memory modules exist later in this track.

In LangChain the messages are typed objects - `SystemMessage`, `HumanMessage`,
`AIMessage` - all importable from `langchain_core.messages`. Let us build a fake
conversation offline first; constructing messages never touches a network.

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# A conversation is just an ordered Python list of these typed message objects.
conversation = [
    # system sets standing rules; most models weight it strongly and persistently.
    SystemMessage(content="You are a concise machine learning tutor."),
    HumanMessage(content="What is a token?"),
    # We replayed a previous ai turn by hand - this is how context gets rebuilt.
    AIMessage(content="A token is the chunk of text a model reads or writes."),
    HumanMessage(content="Give me one example of tokens."),
]

for msg in conversation:
    # __class__.__name__ shows which role each object encodes; ljust keeps columns tidy.
    print(msg.__class__.__name__.ljust(14), "->", msg.content)

SystemMessage 

 -> You are a concise machine learning tutor.
HumanMessage   -> What is a token?
AIMessage      -> A token is the chunk of text a model reads or writes.
HumanMessage   -> Give me one example of tokens.


### 2. Prerequisites and your first real call

One-time setup (outside Python):

1. Install the Ollama app from ollama.com (or `winget install Ollama.Ollama`).
2. Pull the model weights once: `ollama pull llama3.2` downloads about 2 GB.
3. That is all - the app starts a local server on port 11434 automatically.

`ChatOllama(model="llama3.2")` builds a CLIENT object instantly and lazily: no
request happens until you actually invoke it. The string must match the tag shown
by `ollama list` character for character.

> **Common pitfall:** errors surface at `.invoke()`, not at construction. If the
> server is down or the tag is wrong you get an exception only when calling -
> which is why every cell below wraps its calls in a guard that prints setup help
> instead of crashing your notebook run.

In [3]:
from langchain_ollama import ChatOllama

# Build the client. Extra options come later; the model tag is the one required field.
llm = ChatOllama(model="llama3.2")

try:
    # invoke() sends the request and BLOCKS until the full answer is generated.
    # Passing a bare string is shorthand for [HumanMessage("Say hi in five words")].
    response = llm.invoke("Say hi in five words")
    print(response.content)     # .content holds just the text; the object also has metadata.
except Exception:
    # Friendly skip message instead of a traceback, so the notebook still runs top-to-bottom.
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

Hello, how are you today?


### 3. Messages in, message out - for real this time

Now feed the typed-message list from section 1 to the model. The return value is an
`AIMessage`: check its type, read `.content`, and peek inside `.response_metadata`
(model name, eval counts) - useful later for cost/latency dashboards.

> **Pro tip:** the first call after starting Ollama is slow because the model is
> being loaded into RAM/VRAM ("warm-up"). Every subsequent call is much faster,
> so do not judge performance by the very first request.

In [4]:
messages = [
    SystemMessage(content="You answer in exactly one sentence."),  # Standing rule for ALL turns below.
    HumanMessage(content="What is a vector database?"),            # This turn's actual question.
]

try:
    reply = llm.invoke(messages)            # Same client as before, now given structured messages.
    print("Returned type :", type(reply).__name__)      # Expect: AIMessage
    print("Reply content :", reply.content)
    # response_metadata is a dict; sorted(keys) gives a stable, readable preview line.
    print("Metadata keys :", sorted(reply.response_metadata.keys()))
except Exception:
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

Returned type : AIMessage
Reply content : A vector database is a type of data storage and retrieval system that organizes and indexes large collections of numerical vectors, such as those used in machine learning algorithms for text similarity, image recognition, and other applications.
Metadata keys : ['created_at', 'done', 'done_reason', 'eval_count', 'eval_duration', 'load_duration', 'logprobs', 'model', 'model_name', 'model_provider', 'prompt_eval_count', 'prompt_eval_duration', 'total_duration']


### 4. Streaming: watch tokens arrive

`.invoke()` waits for the WHOLE answer before returning anything - fine for scripts,
painful for chat UIs where users stare at a blank screen. `.stream()` returns a
generator that yields message FRAGMENTS as soon as each token is produced, so you can
print progressively and cut perceived latency dramatically.

> **Pro tip:** streaming changes nothing about what the model computes, only how the
> result reaches you. Later, LCEL chains stream end-to-end with the same `.stream()` call.

In [5]:
try:
    start = time.perf_counter()             # High-resolution timer for wall-clock duration.
    n_chunks = 0                            # Count fragments to show streaming granularity.
    for chunk in llm.stream("Count from 1 to 10, separated by commas."):
        # Each chunk is a partial AIMessage; end="" + flush prints them glued together live.
        print(chunk.content, end="", flush=True)
        n_chunks += 1
    print()                                 # Newline after the streamed answer for clean output.
    print(f"[streamed {n_chunks} chunks in {time.perf_counter() - start:.1f} seconds]")
except Exception:
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

Here

's

 the

 count

 from

...

,

9

,

10


[streamed 45 chunks in 0.4 seconds]


### 5. Batch: many prompts, one call site

`.batch()` fans a list of inputs through the same model and returns a list of
`AIMessage` objects IN THE SAME ORDER as the inputs - order preservation is part of
the contract, so index i always matches question i. Under the hood Ollama may process
requests concurrently depending on server settings, but your code does not care.

> **Common pitfall:** batching multiplies token usage and can hit concurrency limits
> on cloud providers (Groq free tier especially). Locally it is free, so batch away.

In [6]:
questions = [
    "Define overfitting in one sentence.",       # Batch item 0
    "Define underfitting in one sentence.",      # Batch item 1
    "Name one classic cure for overfitting.",    # Batch item 2
]

try:
    answers = llm.batch(questions)          # One call site, ordered list of AIMessage back.
    for q, a in zip(questions, answers):    # zip pairs each question with its own answer.
        print("Q:", q)
        print("A:", a.content[:140])        # Truncate long answers to keep the output readable.
        print("-" * 60)                     # ASCII divider between pairs.
except Exception:
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

Q: Define overfitting in one sentence.
A: Overfitting occurs when a model is too complex and learns the noise or random variations in the training data, resulting in poor performance
------------------------------------------------------------
Q: Define underfitting in one sentence.
A: Underfitting occurs when a model is too simple to accurately capture the underlying patterns and relationships in the data, resulting in poo
------------------------------------------------------------
Q: Name one classic cure for overfitting.
A: One classic cure for overfitting is Regularization, specifically L1 or L2 regularization (also known as Ridge regression and Lasso regressio
------------------------------------------------------------


### 6. The temperature experiment

`temperature` rescales how boldly the model picks among likely next tokens:

| Setting | Behavior | Good for |
|---|---|---|
| 0.0 - 0.2 | Near-deterministic, sticks to the safest wording | Extraction, classification, RAG answers in the capstone |
| 0.5 - 0.8 | Balanced variety while staying coherent | General chat, explanations |
| 1.0+ | Surprising word choices, creative but riskier | Brainstorming names, taglines, fiction |

Run the loop a few times: low temperatures repeat almost verbatim across reruns,
high temperatures wander noticeably.

> **Pro tip:** even at `temperature=0` outputs are "mostly stable", not guaranteed
> bit-identical, because GPU kernels are not fully deterministic. Treat reproducibility
> statistically, not absolutely.

In [7]:
PROMPT = "Write a one-sentence tagline for a cozy neighborhood coffee shop."

for temp in (0.0, 0.7, 1.5):
    try:
        # A FRESH client per setting, because temperature is fixed at construction time.
        temp_llm = ChatOllama(model="llama3.2", temperature=temp)
        print(f"--- temperature = {temp} ---")
        print(temp_llm.invoke(PROMPT).content)
    except Exception:
        # Server down or model missing: say so once and leave the loop instead of
        # printing three identical failure messages for the remaining temperatures.
        print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")
        break

--- temperature = 0.0 ---


"Warming hearts, one cup at a time in the coziest corner of town."


--- temperature = 0.7 ---


"Steeped in Community, Brewed with Love."


--- temperature = 1.5 ---


"Warming hearts, one cup at a time in the coziest corner of town."


### 7. Teaser: `init_chat_model`, the universal factory

So far you built clients with a vendor-specific class (`ChatOllama`). LangChain also
ships ONE factory that parses a `"provider:model"` string and returns the right class:

- `"ollama:llama3.2"` returns a `ChatOllama`
- `"groq:openai/gpt-oss-20b"` returns a `ChatGroq` (key required)
- `"openai:gpt-4o-mini"` returns a `ChatOpenAI` (paid key required)

Application code written against the factory NEVER changes when you swap engines -
notebook 04 of this module proves it by running one identical chain on all three.
Keyword arguments like `temperature` forward to whichever class gets constructed.

In [8]:
from langchain.chat_models import init_chat_model

try:
    # Factory call: reads everything BEFORE the colon to pick the provider package.
    local_llm = init_chat_model("ollama:llama3.2", temperature=0.7)
    print("Factory returned class:", type(local_llm).__name__)   # Should print ChatOllama.
    print(local_llm.invoke("Reply with the single word: OK").content)
except Exception:
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

Factory returned class: ChatOllama
OK


### Summary & key takeaways

- A chat model call is **typed messages in, exactly one assistant message out**;
  memory means re-sending history yourself.
- `ChatOllama(model="llama3.2")` gives you a capable open model with zero cost and
  total privacy; failures happen at `invoke()` time, hence the guard pattern.
- `invoke` for completeness, `stream` for responsiveness, `batch` for throughput -
  same model object, three access patterns.
- `temperature` trades determinism for creativity: keep it LOW for the factual
  RAG answering stage of the capstone, raise it only for idea generation.
- `init_chat_model("provider:model")` decouples your code from any single vendor -
  the abstraction the next three notebooks exploit.

**Next up:** notebook 02 swaps the local engine for Groq's blazing-fast cloud inference.